# Notebook 00 — Setup and IFC Exploration

Verifies the environment, installs dependencies, and explores the IFC reference model schema.

**No GPU required.** Run on CPU runtime.

> **Always run Cell 1 first every session.**

In [6]:
# ── Cell 1: Clone, install, set working directory ─────────────────────────────
import os, sys

REPO = 'ifc-graphrag-dt'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
    print('Repository updated.')
else:
    !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git
    print('Repository cloned.')

!bash {REPO}/colab_setup.sh

os.chdir(REPO)
REPO = '.'
print(f'Working directory: {os.getcwd()}')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print('Setup complete.')


Cloning into 'ifc-graphrag-dt'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 134 (delta 52), reused 117 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 121.07 KiB | 10.09 MiB/s, done.
Resolving deltas: 100% (52/52), done.
Repository cloned.
==> Cloning / updating repository...
==> Installing core package in editable mode...
  Preparing metadata (setup.py) ... done
==> Installing ifcopenshell...
==> Installing sentence-transformers and faiss-cpu...
==> Installing scipy and scikit-learn (for statistical tests)...
==> Installing pyyaml, tqdm, rich, jsonschema...

✓ Base setup complete.
  GPU packages (torch, diffusers, TripoSR) are installed only in Notebook 04.
  Run: import sys; sys.path.insert(0, 'ifc-graphrag-dt'); import os; os.chdir('ifc-graphrag-dt')
Working directory: /content/ifc-graphrag-dt/ifc-graphrag-dt
Setup complete.


In [7]:
# ── Cell 2: Verify imports ────────────────────────────────────────────────────
import networkx as nx
import numpy as np
import json
from pathlib import Path

from benchmark.dtah_bench import DTAHBench
from evaluation.metrics.kcs_dt import KCSDTScorer
from evaluation.metrics.ggs import GGSScorer
from pipeline.layer1_retriever.khop_traversal import KHopTraversal

print('All core imports successful.')


All core imports successful.


In [8]:
# ── Cell 3: DTAH-Bench overview ───────────────────────────────────────────────
bench = DTAHBench()
stats = bench.stats()
print('DTAH-Bench statistics:')
for k, v in stats.items():
    print(f'  {k}: {v}')

print('\nSample Tier 1 prompt:')
t1 = bench.load_tier(1)
print(json.dumps(t1[0], indent=2))


DTAH-Bench statistics:
  pilot_mode: False
  tier1_count: 50
  tier2_count: 50
  tier3_count: 50
  total: 150
  domains: ['MEP', 'structural', 'HVAC']

Sample Tier 1 prompt:
{
  "id": "T1-MEP-001",
  "domain": "MEP",
  "prompt": "Generate a centrifugal pump",
  "ifc_entity": "IfcPump",
  "material": "cast_iron",
  "attributes": {
    "flow_rate": "medium",
    "orientation": "horizontal"
  },
  "tier": 1,
  "ifc_relations": []
}


In [9]:
# ── Cell 4: Prompt distribution chart ─────────────────────────────────────────
from collections import Counter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('outputs/figures', exist_ok=True)

all_prompts   = bench.load_all()
tier_counts   = Counter(p['tier'] for p in all_prompts)
domain_counts = Counter(p['domain'] for p in all_prompts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([f'Tier {t}' for t in sorted(tier_counts)],
            [tier_counts[t] for t in sorted(tier_counts)],
            color=['#2E5FA3','#1D9E75','#D85A30'])
axes[0].set_title('Prompts per Tier')
axes[0].set_ylabel('Count')
axes[1].bar(list(domain_counts.keys()), list(domain_counts.values()),
            color=['#2E5FA3','#1D9E75','#D85A30'])
axes[1].set_title('Prompts per Domain')
plt.tight_layout()
plt.savefig('outputs/figures/00_prompt_distribution.png', dpi=150)
plt.show()
print(f'Total prompts: {len(all_prompts)}')


Total prompts: 150


In [11]:
# ── Cell 5: Download IFC reference model ─────────────────────────────────────
from pathlib import Path
import requests

IFC_DIR = Path("benchmark/ifc_reference_models")
DUPLEX_PATH = IFC_DIR / "duplex.ifc"
IFC_DIR.mkdir(parents=True, exist_ok=True)

URLS = [
    # Current buildingSMART community repository
    (
        "https://github.com/buildingsmart-community/"
        "Community-Sample-Test-Files/raw/refs/heads/main/"
        "IFC%202.3.0.1%20%28IFC%202x3%29/Duplex%20Apartment/"
        "Duplex_A_20110907.ifc"
    ),
    # Backup mirror
    (
        "https://raw.githubusercontent.com/stijngoedertier/"
        "georeference-ifc/master/Duplex_A_20110907.ifc"
    ),
]


def file_ok(path):
    """Validate size and IFC STEP-file header."""
    if not path.exists() or path.stat().st_size < 100_000:
        return False

    with path.open("rb") as file:
        header = file.read(256)

    return b"ISO-10303-21" in header


if file_ok(DUPLEX_PATH):
    print(f"Already downloaded: {DUPLEX_PATH} "
          f"({DUPLEX_PATH.stat().st_size / 1024:.0f} KB)")
else:
    downloaded = False

    for index, url in enumerate(URLS, 1):
        print(f"Attempt {index}/{len(URLS)}: {url}")

        try:
            response = requests.get(
                url,
                timeout=120,
                allow_redirects=True,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()

            # Reject HTML pages and Git LFS pointer files.
            if b"ISO-10303-21" not in response.content[:256]:
                raise ValueError(
                    f"Response is not an IFC file "
                    f"({response.headers.get('content-type')}, "
                    f"{len(response.content)} bytes)"
                )

            temp_path = DUPLEX_PATH.with_suffix(".tmp")
            temp_path.write_bytes(response.content)
            temp_path.replace(DUPLEX_PATH)

            print(f"Downloaded: {DUPLEX_PATH.stat().st_size / 1024:.0f} KB")
            downloaded = True
            break

        except Exception as error:
            print(f"  Failed: {error}")

    if not downloaded:
        raise RuntimeError("All IFC download attempts failed.")

print(f"\nFile ready: {DUPLEX_PATH.resolve()}")

Attempt 1/2: https://github.com/buildingsmart-community/Community-Sample-Test-Files/raw/refs/heads/main/IFC%202.3.0.1%20%28IFC%202x3%29/Duplex%20Apartment/Duplex_A_20110907.ifc
Downloaded: 2325 KB

File ready: /content/ifc-graphrag-dt/ifc-graphrag-dt/benchmark/ifc_reference_models/duplex.ifc


In [12]:
# ── Cell 6: Explore IFC schema ────────────────────────────────────────────────
import ifcopenshell

print(f'Opening: {os.path.abspath(DUPLEX_PATH)}')
model = ifcopenshell.open(DUPLEX_PATH)
print(f'IFC schema: {model.schema}')

type_counts = {}
for entity in model:
    t = entity.is_a()
    type_counts[t] = type_counts.get(t, 0) + 1

sorted_types = sorted(type_counts.items(), key=lambda x: -x[1])
print(f'\nTotal entities:      {sum(type_counts.values())}')
print(f'Unique entity types: {len(type_counts)}')
print('\nTop 20 entity types:')
for t, n in sorted_types[:20]:
    print(f'  {t:<45} {n}')


Opening: /content/ifc-graphrag-dt/ifc-graphrag-dt/benchmark/ifc_reference_models/duplex.ifc
IFC schema: IFC2X3

Total entities:      38898
Unique entity types: 103

Top 20 entity types:
  IfcCartesianPoint                             8520
  IfcPropertySingleValue                        5213
  IfcPolyLoop                                   4546
  IfcFace                                       4486
  IfcFaceOuterBound                             4486
  IfcRelDefinesByProperties                     1480
  IfcPropertySet                                1459
  IfcAxis2Placement3D                           1279
  IfcPolyline                                   864
  IfcShapeRepresentation                        483
  IfcExtrudedAreaSolid                          421
  IfcAxis2Placement2D                           397
  IfcCompositeCurveSegment                      322
  IfcLocalPlacement                             295
  IfcRectangleProfileDef                        293
  IfcProductDefinitionShap

In [13]:
# ── Cell 7: IFC relations tracked by the pipeline ─────────────────────────────
from pipeline.layer1_retriever.ifc_graph_builder import IFC_RELATION_CATEGORIES

print('IFC relations tracked by IFC-GraphRAG-DT:')
print(f'{"Relation Type":<48} {"Category":<25} Instances')
print('-' * 88)
for rel, cat in IFC_RELATION_CATEGORIES.items():
    count = len(model.by_type(rel))
    print(f'{rel:<48} {cat:<25} {count}')


IFC relations tracked by IFC-GraphRAG-DT:
Relation Type                                    Category                  Instances
----------------------------------------------------------------------------------------
IfcRelContainedInSpatialStructure                containment               15
IfcRelAggregates                                 aggregation               9
IfcRelConnects                                   connectivity              450
IfcRelConnectsPortToElement                      port_connectivity         0
IfcRelAssignsToGroup                             system_membership         0
IfcRelAssigns                                    assignment                0
IfcRelVoidsElement                               void                      50
IfcRelFillsElement                               fill                      38
IfcRelAssociatesMaterial                         material                  92
IfcRelDefinesByProperties                        property                  1480
IfcRe

In [14]:
# ── Cell 8: KCS-DT smoke test ─────────────────────────────────────────────────
scorer = KCSDTScorer()
gt = {
    'entities':    [{'ifc_type': 'IfcPump'}, {'ifc_type': 'IfcPipeSegment'}, {'ifc_type': 'IfcValve'}],
    'relations':   [{'type': 'IfcRelConnectsPortToElement', 'from': 'IfcPump.OutletPort', 'to': 'IfcPipeSegment'},
                    {'type': 'IfcRelConnects', 'from': 'IfcPipeSegment', 'to': 'IfcValve'}],
    'attributes':  {'pump_0': {'material': 'cast_iron'}},
    'containment': [{'entity': 'IfcPump', 'container': 'IfcSpace'}],
    'connectivity':[{'from': 'IfcPump.OutletPort', 'to': 'IfcPipeSegment'}]
}
perfect = scorer.score(gt, gt, prompt_id='smoke-test')
print(f'KCS-DT perfect score: {perfect}')
assert abs(perfect.total - 1.0) < 1e-4, f'Expected 1.0, got {perfect.total}'

empty_pred = {'entities':[],'relations':[],'attributes':{},'containment':[],'connectivity':[]}
empty = scorer.score(empty_pred, gt)
print(f'KCS-DT empty pred:    {empty}')
assert empty.entity == 0.0
print('KCS-DT smoke test PASSED.')


KCS-DT perfect score: KCSDTScore(E=1.000, R=1.000, A=1.000, Cn=1.000, Cv=1.000 → total=1.000)
KCS-DT empty pred:    KCSDTScore(E=0.000, R=0.000, A=0.000, Cn=0.000, Cv=0.000 → total=0.000)
KCS-DT smoke test PASSED.


In [15]:
# ── Cell 9: GGS smoke test ────────────────────────────────────────────────────
ggs_scorer = GGSScorer()

G = nx.DiGraph()
G.add_node('A', ifc_type='IfcPump',        name='P-01')
G.add_node('B', ifc_type='IfcPipeSegment', name='Pipe-01')
G.add_node('C', ifc_type='IfcValve',       name='V-01')
G.add_edge('A', 'B', relation_type='IfcRelConnectsPortToElement')
G.add_edge('B', 'C', relation_type='IfcRelConnects')

ggs = ggs_scorer.score(G, G, prompt_id='smoke-test')
print(f'GGS perfect score: {ggs}')
assert abs(ggs.node_recall - 1.0) < 1e-4
assert abs(ggs.edge_recall - 1.0) < 1e-4
print('GGS smoke test PASSED.')
print('\n' + '='*55)
print('All checks passed. Ready for Notebook 01.')
print('='*55)


GGS perfect score: GGSScore(N=1.000, E=1.000, P=1.000 → total=1.000)
GGS smoke test PASSED.

All checks passed. Ready for Notebook 01.
